# Notebook de démarrage

Objectif : générer un dataset synthétique (classification ou régression), effectuer un EDA rapide et entraîner un pipeline scikit-learn minimal.

Instructions : exécutez les cellules dans l'ordre. Modifiez les paramètres du générateur si vous voulez tester différents scénarios.

In [ ]:
# Imports de base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error

sns.set(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
# Générateur de données synthétiques
def generate_synthetic_data(kind='classification', n_samples=1000, n_features=10, noise=0.1, random_state=RANDOM_STATE):
    import sklearn.datasets as ds
    if kind == 'classification':
        X, y = ds.make_classification(n_samples=n_samples, n_features=n_features, n_informative=int(n_features/2), n_redundant=0, n_clusters_per_class=1, flip_y=noise, random_state=random_state)
    else:
        X, y = ds.make_regression(n_samples=n_samples, n_features=n_features, n_informative=int(n_features/2), noise=noise*10, random_state=random_state)
    cols = [f'feat_{i}' for i in range(X.shape[1])]
    df = pd.DataFrame(X, columns=cols)
    df['target'] = y
    return df

# Exemple : générer un dataset de classification
df = generate_synthetic_data(kind='classification', n_samples=1000, n_features=8)
df.head()

In [ ]:
# EDA rapide
print('Shape:', df.shape)
display(df.describe().T)

# Distribution de la target (classification)
if df['target'].dtype.kind in 'fi':
    # classification generator encodera la target comme 0/1 si classification
    try:
        sns.countplot(x='target', data=df)
        plt.title('Distribution de la target')
        plt.show()
    except Exception:
        print('Impossible d'afficher countplot pour la target')

# Corrélations rapides
corr = df.corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Matrice de corrélation (features + target)')
plt.show()

In [ ]:
# Pipeline minimal d'exemple (classification)
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('Classification report:')
print(classification_report(y_test, y_pred))